# Pandas vs Polars: A Head-to-Head Comparison

**Prerequisites**

- Pandas (L02-L07)
- Polars introduction

**Outcomes**

- See concrete performance differences between pandas and Polars on real data
- Understand *why* Polars is faster (parallelism, lazy evaluation, query optimization)
- Build intuition for when each library is the right tool

In [ ]:
import glob

import pandas as pd
import polars as pl

## The Question

We spent most of this semester using pandas. It served us well for datasets with thousands or even hundreds of thousands of rows. But now we have AIS ship tracking data with **7 million rows per file** and hundreds of files.

How much does our choice of tool matter?

Let's find out by running the exact same analysis in both pandas and Polars, step by step, with timing.

## Step 1: Loading a Parquet File

The simplest operation -- read a single day of AIS data from a parquet file.

In [ ]:
ais_file = "data/2024/ais-2024-01-01.parquet"

### Pandas

In [ ]:
%%time

df_pd = pd.read_parquet(ais_file)
print(f"Shape: {df_pd.shape}")

### Polars

In [ ]:
%%time

df_pl = pl.read_parquet(ais_file)
print(f"Shape: {df_pl.shape}")

Same data, same file format, but Polars loads it significantly faster. This is because Polars uses multiple CPU cores to read and decompress the parquet file in parallel, while pandas is limited to a single core.

## Step 2: Parsing Timestamps

The `base_date_time` column is stored as a string. We need to convert it to a proper datetime type before we can do any time-based analysis.

### Pandas

In [ ]:
%%time

df_pd["timestamp"] = pd.to_datetime(
    df_pd["base_date_time"], format="%Y-%m-%dT%H:%M:%S"
)

### Polars

In [ ]:
%%time

df_pl = df_pl.with_columns(
    pl.col("base_date_time")
    .str.to_datetime("%Y-%m-%dT%H:%M:%S")
    .alias("timestamp")
)

String parsing is one of the most expensive common operations in data work. Polars parallelizes this across cores, giving a large speedup.

## Step 3: Filtering

Let's filter to cargo ships (vessel type 70-79) that are effectively stopped (speed < 0.5 knots).

### Pandas

In [ ]:
%%time

stopped_pd = df_pd[
    (df_pd["vessel_type"] >= 70)
    & (df_pd["vessel_type"] <= 79)
    & (df_pd["sog"] < 0.5)
]

print(f"Rows after filter: {len(stopped_pd):,}")

### Polars

In [ ]:
%%time

stopped_pl = df_pl.filter(
    pl.col("vessel_type").is_between(70, 79),
    pl.col("sog") < 0.5,
)

print(f"Rows after filter: {stopped_pl.shape[0]:,}")

Filtering is fast in both libraries. The difference is smaller here because the operation is simple (just comparing numbers). However, notice how the Polars syntax is arguably cleaner -- the `.is_between` method and the ability to pass multiple filter conditions as separate arguments.

## Step 4: Group By + Aggregation

Now let's group by ship (`mmsi`) and compute the number of stopped observations and mean location.

### Pandas

In [ ]:
%%time

result_pd = stopped_pd.groupby("mmsi").agg(
    n_stopped=("timestamp", "count"),
    mean_lat=("latitude", "mean"),
    mean_lon=("longitude", "mean"),
)

port_ships_pd = result_pd[result_pd["n_stopped"] >= 360]
print(f"Ships stopped 6+ hours: {len(port_ships_pd)}")

### Polars

In [ ]:
%%time

port_ships_pl = (
    stopped_pl
    .group_by("mmsi")
    .agg(
        pl.col("timestamp").count().alias("n_stopped"),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .filter(pl.col("n_stopped") >= 360)
)

print(f"Ships stopped 6+ hours: {port_ships_pl.shape[0]}")

On this step the timings are close. The group-by is operating on the *filtered* data (~300k rows), which is small enough that both libraries handle it quickly.

The real question is: what happens when we put it all together?

## The Full Pipeline: Where It Really Matters

In practice, we don't run steps in isolation. We load data, parse, filter, group, and aggregate as a single pipeline.

This is where Polars' **lazy evaluation** changes the game. Instead of executing each step immediately (like pandas does), Polars builds a query plan and optimizes the entire pipeline before running anything.

Let's compare the full end-to-end pipeline on **three days** of data.

In [ ]:
files = sorted(glob.glob("data/2024/ais-2024-01-0[1-3].parquet"))
print(f"Files: {files}")

### Pandas: Eager, Step-by-Step

In [ ]:
%%time

# Step 1: Load all three files and concatenate
dfs = [pd.read_parquet(f) for f in files]
combined_pd = pd.concat(dfs, ignore_index=True)

# Step 2: Parse timestamps
combined_pd["timestamp"] = pd.to_datetime(
    combined_pd["base_date_time"], format="%Y-%m-%dT%H:%M:%S"
)

# Step 3: Filter to stopped cargo ships
stopped_pd = combined_pd[
    (combined_pd["vessel_type"] >= 70)
    & (combined_pd["vessel_type"] <= 79)
    & (combined_pd["sog"] < 0.5)
]

# Step 4: Group by ship and find those stopped 6+ hours
result_pd = stopped_pd.groupby("mmsi").agg(
    n_stopped=("timestamp", "count"),
    mean_lat=("latitude", "mean"),
    mean_lon=("longitude", "mean"),
)
result_pd = result_pd[result_pd["n_stopped"] >= 360]

print(f"Ships stopped 6+ hours: {len(result_pd)}")

### Polars: Lazy, Optimized

Notice the key difference: we use `pl.scan_parquet` (not `read_parquet`) and chain all operations into a single query. Polars doesn't execute anything until we call `.collect()` at the end.

In [ ]:
%%time

result_pl = (
    pl.scan_parquet(files)
    .with_columns(
        pl.col("base_date_time")
        .str.to_datetime("%Y-%m-%dT%H:%M:%S")
        .alias("timestamp")
    )
    .filter(
        pl.col("vessel_type").is_between(70, 79),
        pl.col("sog") < 0.5,
    )
    .group_by("mmsi")
    .agg(
        pl.col("timestamp").count().alias("n_stopped"),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .filter(pl.col("n_stopped") >= 360)
    .collect()
)

print(f"Ships stopped 6+ hours: {result_pl.shape[0]}")

Same answer. Dramatically less time.

The speedup on the full pipeline is much larger than what we saw on any individual step. This is because Polars can **optimize across steps**.

## Why is the Full Pipeline So Much Faster?

Three reasons:

### 1. Predicate Pushdown

When Polars sees that we filter on `vessel_type` and `sog`, it can push those filters down into the parquet reader itself. This means Polars **skips reading columns and row groups that don't match the filter**.

Pandas, by contrast, must read *all* 7+ million rows into memory and *then* filter.

### 2. Projection Pushdown

Polars notices which columns are actually used downstream and only reads those columns from the parquet file. If we never use `heading` or `call_sign`, they are never loaded.

### 3. Parallelism

Every step -- reading, parsing, filtering, grouping -- uses all available CPU cores.

We can see the optimized query plan by calling `.explain()` before `.collect()`.

In [ ]:
query = (
    pl.scan_parquet(files)
    .with_columns(
        pl.col("base_date_time")
        .str.to_datetime("%Y-%m-%dT%H:%M:%S")
        .alias("timestamp")
    )
    .filter(
        pl.col("vessel_type").is_between(70, 79),
        pl.col("sog") < 0.5,
    )
    .group_by("mmsi")
    .agg(
        pl.col("timestamp").count().alias("n_stopped"),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .filter(pl.col("n_stopped") >= 360)
)

print(query.explain())

Look for the `FILTER` and `PROJECT` lines near the bottom of the plan (closest to the data source). These show Polars pushing operations as close to the raw data as possible.

## Scaling the Comparison

Let's see how the gap changes as we increase the number of files. We'll time the full pipeline for 1, 3, 5, and 7 days.

In [ ]:
import time

all_jan_files = sorted(glob.glob("data/2024/ais-2024-01-0[1-7].parquet"))

results = []

for n_days in [1, 3, 5, 7]:
    files_subset = all_jan_files[:n_days]

    # --- Pandas ---
    t0 = time.time()
    dfs = [pd.read_parquet(f) for f in files_subset]
    combined = pd.concat(dfs, ignore_index=True)
    combined["timestamp"] = pd.to_datetime(
        combined["base_date_time"], format="%Y-%m-%dT%H:%M:%S"
    )
    stopped = combined[
        (combined["vessel_type"] >= 70)
        & (combined["vessel_type"] <= 79)
        & (combined["sog"] < 0.5)
    ]
    res = stopped.groupby("mmsi").agg(
        n_stopped=("timestamp", "count"),
        mean_lat=("latitude", "mean"),
        mean_lon=("longitude", "mean"),
    )
    _ = res[res["n_stopped"] >= 360]
    t_pandas = time.time() - t0

    # --- Polars (lazy) ---
    t0 = time.time()
    _ = (
        pl.scan_parquet(files_subset)
        .with_columns(
            pl.col("base_date_time")
            .str.to_datetime("%Y-%m-%dT%H:%M:%S")
            .alias("timestamp")
        )
        .filter(
            pl.col("vessel_type").is_between(70, 79),
            pl.col("sog") < 0.5,
        )
        .group_by("mmsi")
        .agg(
            pl.col("timestamp").count().alias("n_stopped"),
            pl.col("latitude").mean().alias("mean_lat"),
            pl.col("longitude").mean().alias("mean_lon"),
        )
        .filter(pl.col("n_stopped") >= 360)
        .collect()
    )
    t_polars = time.time() - t0

    results.append({
        "n_days": n_days,
        "pandas_sec": round(t_pandas, 1),
        "polars_sec": round(t_polars, 2),
        "speedup": f"{t_pandas / t_polars:.0f}x",
    })

    print(f"{n_days} day(s): pandas={t_pandas:.1f}s, polars={t_polars:.2f}s ({t_pandas/t_polars:.0f}x)")

pl.DataFrame(results)

The speedup tends to *increase* as we add more data. This is because:

1. Polars' query optimization saves more work on larger datasets (more rows to skip)
2. Polars can read multiple files in parallel
3. Pandas' memory overhead grows linearly, while Polars streams data through

## Visualizing the Comparison

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("ggplot")

df_results = pl.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: absolute times
axes[0].plot(
    df_results["n_days"].to_numpy(),
    df_results["pandas_sec"].to_numpy(),
    "o-", label="pandas", linewidth=2, markersize=8
)
axes[0].plot(
    df_results["n_days"].to_numpy(),
    df_results["polars_sec"].to_numpy(),
    "s-", label="polars (lazy)", linewidth=2, markersize=8
)
axes[0].set_xlabel("Number of daily files")
axes[0].set_ylabel("Seconds")
axes[0].set_title("Total Pipeline Time")
axes[0].legend()

# Right: speedup factor
speedups = [
    p / q
    for p, q in zip(
        df_results["pandas_sec"].to_list(),
        df_results["polars_sec"].to_list(),
    )
]
axes[1].bar(
    df_results["n_days"].to_numpy(),
    speedups,
    width=0.6, alpha=0.7
)
axes[1].set_xlabel("Number of daily files")
axes[1].set_ylabel("Speedup (pandas time / polars time)")
axes[1].set_title("Polars Speedup Factor")

fig.suptitle("Full AIS Pipeline: pandas vs Polars", fontsize=14, y=1.02)
fig.tight_layout()

## When Should You Use Pandas?

Despite these results, pandas is not "worse" than Polars in every situation. Pandas is still a great choice when:

- **Your data is small** (< 1 million rows). At this scale, the overhead of Polars' query planner may not pay off, and pandas' mature API is very convenient.
- **You need ecosystem compatibility**. Many libraries (statsmodels, seaborn, scikit-learn) expect pandas DataFrames as input. You can convert with `.to_pandas()`, but if your entire workflow is in the pandas ecosystem, there may be little reason to switch.
- **You are doing exploratory, interactive work**. Pandas' eager execution gives you immediate feedback, which can be more intuitive when you're still figuring out what questions to ask.
- **You need the most stable, battle-tested option**. Pandas has been around since 2008 and is used in production at thousands of companies. Polars is newer and its API still evolves.

The sweet spot for Polars is when your data is large enough that performance matters and your pipeline is well-defined enough to benefit from lazy evaluation -- exactly the situation we find ourselves in with AIS data.

## Summary

| | pandas | Polars |
|---|---|---|
| **Execution** | Eager (step-by-step) | Lazy (builds query plan, then optimizes) |
| **Parallelism** | Single-core | Multi-core by default |
| **File reading** | Reads all columns and rows | Pushes filters/projections into the reader |
| **Memory** | Loads entire dataset into RAM | Streams data, discards what it doesn't need |
| **API style** | `df["col"]`, `df.query()`, `df.groupby()` | `pl.col("col")`, `.filter()`, `.group_by()` |
| **Best for** | Small-medium data, exploration | Large data, production pipelines |